---
# Chapter 3 — The RAG Baseline

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 3: The RAG Baseline |
| Central question | How far can a well-built conventional RAG system take us before additional memory machinery earns its cost? |
| Main concepts | Conventional RAG baseline, Hybrid retrieval, Context assembly, Source-preserving boundary, Reader use |
| Implementation | memory_baseline (PostgreSQL, pgvector, hybrid retrieval, reranking, context assembly) |
| Experiment | ch3-20260919-ladder (8 conditions) |
| Evidence status | Implemented; comparative book experiment pending |
| Depends on | Chapter 2 (instrument), Chapter 1 (definitions) |

---

## What this notebook demonstrates

This chapter builds the **strongest conventional RAG baseline** — the serious rival every later memory mechanism must beat. The notebook:

1. **Loads the actual `memory_baseline` implementation** (PostgreSQL + pgvector + hybrid retrieval + reranking + context assembly)
2. **Shows the pipeline**: discover → parse → chunk → embed → index → retrieve → fuse → rerank → assemble → read
3. **Inspects candidates and admitted context** via `ContextTrace`
4. **Loads the frozen 8-condition ladder run** (`ch3-20260919-ladder`)
5. **Demonstrates that strong RAG is a serious baseline** — not a strawman

> **Evidence status**: The baseline is implemented with development runs. The comparative book experiment is pending due to fixture, label, and scoring-stage defects (see Chapter 3 limitations). The frozen ladder run exists but uses a small hand-authored fixture.

## The chapter question

> **How far can a well-built conventional RAG system take us before additional memory machinery earns its cost?**

The baseline keeps **source records, chunks, ordinary metadata, and search indexes** between queries. It does **not** maintain its own interpreted account of decisions or current beliefs. A field declaring "this is the current authoritative decision" would be an interpretation — the baseline only reads what the source text says.

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(3)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 3 Concepts")

## The running example

Same canonical history from Chapter 1:

- **Event-store decision**: SQLite → PostgreSQL via `adr-007` (Jul 11)
- **Rejected cache**: Redis proposed in `session-040`, rejected in `adr-009` (Aug 12)
- **Superseded fact**: Production ran SQLite until Jul 22, then PostgreSQL

The baseline must: retrieve the right passages, admit them to context, and let the reader distinguish proposal from decision, current from historical.

## The mechanism: The Baseline Pipeline

```text
Project artifacts
    ↓ discover, parse, retain provenance, chunk
PostgreSQL
    ├── source records and chunk text
    ├── full-text index
    └── pgvector embeddings and optional ANN index
                  ↓
Query → lexical candidates + dense candidates
                  ↓
             rank fusion (RRF)
                  ↓
        query–passage reranking
                  ↓
       deduplicate and admit context
                  ↓
           reader → answer + citations
```

In [ ]:
# Load the actual baseline implementation
from memory_baseline.pipeline import Baseline, AskResult, RefreshReport
from memory_baseline.config import (
    BaselineConfig,
    ChunkingConfig,
    EmbeddingConfig,
    RetrievalConfig,
    ContextConfig,
    GeneratorConfig,
)
from memory_baseline.embeddings import EmbeddingProvider

print("memory_baseline modules loaded successfully")
print(f"BaselineConfig fields: {list(BaselineConfig.__dataclass_fields__.keys())}")

## Inspect the configuration (every field lands in the run manifest)

In [ ]:
# Show default configuration
config = BaselineConfig()

print("=== Chunking ===")
for k, v in config.chunking.__dict__.items():
    print(f"  {k}: {v}")

print("\n=== Embedding ===")
for k, v in config.embedding.__dict__.items():
    print(f"  {k}: {v}")

print("\n=== Retrieval ===")
for k, v in config.retrieval.__dict__.items():
    print(f"  {k}: {v}")

print("\n=== Context ===")
for k, v in config.context.__dict__.items():
    print(f"  {k}: {v}")

print("\n=== Generator ===")
for k, v in config.generator.__dict__.items():
    if k != 'system_prompt':
        print(f"  {k}: {v}")
    else:
        print(f"  system_prompt: {v[:80]}...")

## The chunking decision

Chunking decides what travels together. The implementation offers three policies:

1. **Fixed character windows** with overlap
2. **Sentence packing** around target size (default: 2000 chars, 300 overlap)
3. **Markdown heading boundaries**

The chapter shows why this matters: if a passage ends after the Redis proposal but the rejection is in the next chunk, retrieval may find the proposal and omit the outcome.

In [ ]:
# Demonstrate chunking on a sample text
from memory_baseline.ingest import chunk_source
from memory_baseline.config import ChunkingConfig
from memory_baseline.ingest import Source, sha1

sample_text = """# Session 040 - Redis Proposal

We should introduce Redis for caching. It would improve read latency significantly.

# Session 044 - Redis Rejection

After evaluation, Redis is unnecessary. Our current PostgreSQL setup handles the load.

# ADR-009 - Redis Rejection Record

DECIDED: Do not introduce Redis. The complexity cost outweighs benefits."""

source = Source(
    source_id="demo-session.md",
    artifact_type="session",
    content=sample_text,
    content_hash=sha1(sample_text),
    timestamp="2024-08-01",
)

for policy in ["fixed", "sentence", "section"]:
    cfg = ChunkingConfig(policy=policy, target_chars=500, overlap_chars=100)
    chunks = chunk_source(source, cfg)
    print(f"\n=== Policy: {policy} ({len(chunks)} chunks) ===")
    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i}: {chunk.text[:100]}...")

## Retrieval paths: Lexical + Dense + Fusion

- **Lexical**: PostgreSQL `to_tsvector` + `ts_rank_cd` (OR combination of query terms)
- **Dense**: pgvector cosine similarity (BGE-M3 default, 1024-dim)
- **Fusion**: Reciprocal Rank Fusion (k=60) — combines rankings without comparing scores across scales
- **Reranker**: Cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) on top 8 candidates

In [ ]:
# Show the retrieval configuration options
from memory_baseline.config import RetrievalConfig

print("Retrieval modes:")
print("  lexical  - PostgreSQL full-text search only")
print("  dense    - pgvector cosine similarity only")
print("  hybrid   - RRF fusion of both (default)")
print()
print("Reranker options:")
print("  none          - no reranking")
print("  cross-encoder - cross-encoder/ms-marco-MiniLM-L-6-v2 (default)")
print("  overlap       - lexical overlap heuristic")
print()
print("Default candidate_k: 30 per path")
print("Default rerank_k: 8")
print("Default fusion_k (RRF constant): 60")

## Context assembly and ContextTrace

`ContextTrace` records admitted passages, duplicate drops, budget drops, character count, estimated token count, and source coverage. Assembly walks ranked list, suppresses exact duplicates, applies passage and character limits.

In [ ]:
from memory_baseline.context import ContextConfig, assemble
from memory_baseline.storage import ScoredChunk

# Simulate ranked candidates exactly as the retriever returns them
texts = [
    ("adr-007", "DECIDED: New event-store work targets PostgreSQL. SQLite prototype superseded."),
    ("session-035", "Benchmark: PostgreSQL handles 10x concurrent writes. Incident reported from production."),
    ("session-031", "Team discusses event log backend. Proposal: use SQLite for simplicity."),
    ("adr-009", "DECIDED: Do not introduce Redis. The complexity cost outweighs benefits."),
]
ranked = [
    ScoredChunk(chunk_id=f"{sid}-{i}", source_id=sid, text=t,
                section=None, score=0.95 - 0.07 * i, rank=i + 1)
    for i, (sid, t) in enumerate(texts)
]

trace = assemble(ranked, ContextConfig(max_chars=2000, max_passages=3))

print("=== ContextTrace (real assemble) ===")
print(f"Admitted passages: {len(trace.admitted)}")
print(f"Admitted chars: {trace.admitted_chars}")
print(f"Estimated tokens: {trace.admitted_tokens_estimate}")
print(f"Sources covered: {trace.sources_covered}")
print(f"Dropped duplicates: {trace.dropped_duplicates}")
print(f"Dropped over budget: {trace.dropped_over_budget}")
print()
for chunk in trace.admitted:
    print(f"  [rank {chunk.rank}] {chunk.source_id}: {chunk.text[:80]}...")
print()
print("Rendered context head:")
print(trace.render()[:300])

## The core ask path

The baseline's `ask` method is short and inspectable:

In [ ]:
import inspect

from memory_baseline.pipeline import Baseline

print(inspect.getsource(Baseline.ask))

## Load the frozen ladder run

The chapter ran an 8-condition ladder over the fixture corpus. Let's load the frozen results.

In [ ]:
from notebooks.memory._support import load_frozen_run

run = load_frozen_run("ch3-20260919-ladder")
summary = run["summary"]

print(f"Run ID: {summary['run_id']} ({summary['task_set_version']}, "
      f"{summary['corpus_version']}, commit {summary['code_commit']})")
render_table(
    [{"Condition": cond,
      "Decision exactness": m["means"]["decision_exactness"],
      "Source recall": m["means"]["source_recall"],
      "Source precision": round(m["means"]["source_precision"], 3),
      "Abstention": m["means"]["abstention_correctness"]}
     for cond, m in summary["ladder"].items()],
    "Ch3 ladder means (frozen, llama3.1:8b)")

## Inspect individual condition results

The ladder conditions:

| Condition | What it isolates |
|-----------|------------------|
`| No supplied history | What the reader answers without project evidence |
`,`| Lexical | What word matching contributes |
`,`| Dense | What the embedding path contributes |
`,`| Hybrid | What combining candidate paths contributes |
`,`| Hybrid + reranker | What the second-stage ranking contributes |
`,`| Best configured | The reference condition for later mechanisms |
`,`| Full history | Whether selection helps vs reading everything |
`,`| Oracle evidence | What the reader does with evaluator-selected sources |


In [ ]:
print("=== Condition Details (from committed manifests) ===\n")
for cond_name, details in sorted(run.get("condition_details", {}).items()):
    manifest = details.get("manifest", {})
    cases = details.get("cases", [])
    print(f"--- {cond_name} ---")
    print(f"  Manifest condition: {manifest.get('condition', 'N/A')}")
    print(f"  Memory configuration: {manifest.get('memory_configuration', 'N/A')}")
    print(f"  Reader model: {manifest.get('model_version', 'N/A')}")
    print(f"  Corpus: {manifest.get('corpus_version', 'N/A')} (commit {manifest.get('code_commit', 'N/A')})")
    print(f"  Cases: {len(cases)}")
    if cases:
        first = cases[0]
        print(f"  Example task: {first.get('task_id', 'N/A')}")
        print(f"  Retrieved: {first.get('retrieved_source_ids', [])}")
        print(f"  Admitted: {first.get('admitted_source_ids', [])}")
        print(f"  Context tokens (est.): {first.get('context_tokens_estimate', 'N/A')}")
    print()

## What the existing runs establish — and what they do not

From the chapter:

> **Development artifacts exist** for embedding comparisons, chunking comparisons, the eight-condition ladder, reader comparisons, and context sweeps. They do **not yet supply an admissible comparative book result**: the history is a small hand-authored fixture rather than the controlled-world generator or a separately adjudicated real corpus, with no genuine unfinished-work evaluation and no executed downstream service-building test. Known fixture, scoring-stage, and labelling defects are recorded in the research notes for versioned follow-ups. The correct status is **implemented; comparative book experiment pending**.

The frozen run `ch3-20260919-ladder` is a **development run**, not a publication-grade frozen experiment.

## Reading a failure without inventing its cause

The chapter provides a diagnostic table for tracing failures through the pipeline:

In [ ]:
failure_table = [
    {"Observation": "Necessary evidence absent from exposed corpus", "First place to investigate": "Collection, parsing, or task answerability"},
    {"Observation": "Evidence indexed but absent from candidates", "First place to investigate": "Lexical matching, embedding, chunking, filters, ANN search"},
    {"Observation": "Evidence is candidate but ranks below distractors", "First place to investigate": "Fusion and reranking"},
    {"Observation": "Ranks adequately but not admitted", "First place to investigate": "Duplication, truncation, context limits"},
    {"Observation": "Reaches context but answer misreads it", "First place to investigate": "Interpretation by the reader"},
    {"Observation": "States constraint but action ignores it", "First place to investigate": "Downstream evidence use"},
    {"Observation": "Stale source displaces relevant current source", "First place to investigate": "Stage of displacement, then temporal interpretation"},
    {"Observation": "Extra context makes answer worse", "First place to investigate": "Matched removal or ordering experiment"},
    {"Observation": "Output defensible but scorer rejects it", "First place to investigate": "Labels, normalisation, evaluation contract"},
]

render_table(failure_table, "Failure Diagnosis Table (Chapter 3)")

## What this establishes

- **Conventional RAG is the rival to beat** — hybrid retrieval + reranking + context assembly + capable reader
- **Retrieval proposes; context decides** — candidate recall ≠ admitted evidence
- **Reader strength, context allowance, retrieval quality are separable** — must test with matched evidence, fixed budgets, oracle controls
- **Source-preserving boundary**: Every answer traceable to which part of which source produced the passage
- **The baseline remains valuable even when another mechanism wins** — raw retrieval is the fallback route to historical record

## What this does NOT establish

- No comparative book verdict (fixture, label, scoring-stage defects)
- No frozen comparative runs claimed
- The baseline is a **functional baseline**, not a permanent commitment to one model
- Chunking policies are configuration, not empirically established optima

## Try it yourself

If you have PostgreSQL running with the baseline schema, you can run a live query (requires `RUN_LIVE = True` and database setup):

In [ ]:
# TRY IT YOURSELF: Configure a live run (commented out by default)
RUN_LIVE = False  # Set to True if you have PostgreSQL + Ollama running

if RUN_LIVE:
    # This would require:
    # 1. PostgreSQL at memory_baseline DSN
    # 2. Ollama with bge-m3 and llama3.1:8b
    # 3. A corpus directory with project history
    print("Live run configured. Set RUN_LIVE=True and ensure services are running.")
else:
    print("Live run disabled (default). The notebook uses frozen artifacts by default.")
    print("To enable: set RUN_LIVE=True and ensure PostgreSQL + Ollama are available.")

## Where this leads next

Chapter 4 asks: **What changes when the question moves from finding a discussion to identifying its outcome?**

With a working retrieval pipeline, six-question instrument, and source-preserving boundary, the next comparison tests whether **persistent derived structure** (GraphRAG) improves diagnosed failures enough to justify its cost.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory-chapter.ipynb)